In [1]:
# ================================================================
# BASELINE LATENCY BENCHMARK — EfficientNet-B7 + LSTM (FULL SEQUENCE)
# Same 5 DAiSEE snippets & protocol as the proposed-model run
# (20 warm-up + 100 timed per snippet = 500 trials per run)
# Each trial: T=10 frames -> EfficientNet-B7 -> LSTM -> classifier
# Reproducibility: fixed seed, fixed weights, fixed inputs, fixed
# threads, output fingerprint check, repeated runs + run log
# ================================================================
import os, time, warnings, sys, platform, subprocess, random, gc, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from torchvision.models import efficientnet_b7
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")

# ── Reproducibility settings ────────────────────────────────────
SEED       = 42
N_THREADS  = 4           # fixed CPU thread count (keep the same every run)
N_REPEATS  = 3           # full benchmark repeats to measure run-to-run variation

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(N_THREADS)
cv2.setNumThreads(1)

# ── Paths ───────────────────────────────────────────────────────
TEST_VIDEO_DIR = "/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test"
OUT_DIR        = "/Users/mike/Desktop/Praxis/DAiSEE/benchmark"
os.makedirs(OUT_DIR, exist_ok=True)
RUN_LOG = os.path.join(OUT_DIR, "rq1_run_log.csv")

# ── Same 5 clips selected in the proposed-model run ─────────────
SNIPPETS = [
    ("Snippet 1: Engaged",             "5100451035.avi"),
    ("Snippet 2: Passive",             "5100401039.avi"),
    ("Snippet 3: Disengaged",          "5100462003.avi"),
    ("Snippet 4: Bright illumination", "8264120146.avi"),
    ("Snippet 5: Dark illumination",   "9070011010.mp4"),
]

# ── Hyper-parameters ────────────────────────────────────────────
T           = 10
IMG_SIZE    = 224
NUM_CLASSES = 3
LSTM_HIDDEN = 512        # set to match your baseline training code
N_WARMUP    = 20
N_TIMED     = 100
TARGET      = 130.0      # ms
FORCE_CPU   = False      # True = run baseline on CPU

# ── Proposed model: 500-trial run (latency-p95-finalv3-simpleone) ─
PROPOSED = dict(mean=110.69, median=107.89, std=20.48,
                p95=117.22, p99=138.67, params=6_574_083)

# ── Hardware ────────────────────────────────────────────────────
def detect_cpu():
    try:
        return subprocess.check_output(
            ["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
    except Exception:
        return platform.processor() or "unknown CPU"

if FORCE_CPU or not torch.backends.mps.is_available():
    DEVICE = torch.device("cpu")
else:
    DEVICE = torch.device("mps")
BACKEND = "MPS (Metal GPU)" if DEVICE.type == "mps" else "CPU"
HARDWARE_STR = f"{detect_cpu()}, {BACKEND}"
print(f"Hardware: {HARDWARE_STR} | torch {torch.__version__} | "
      f"Python {platform.python_version()} | seed {SEED} | threads {N_THREADS}")

def sync():
    if DEVICE.type == "mps":
        torch.mps.synchronize()

# ================================================================
# STEP 1: LOCATE THE 5 SNIPPETS
# ================================================================
VIDEO_INDEX = {}
for root, _, files in os.walk(TEST_VIDEO_DIR):
    for f in files:
        if f.lower().endswith((".avi", ".mp4")):
            VIDEO_INDEX[f] = os.path.join(root, f)

selected = []
for name, clip in SNIPPETS:
    path = VIDEO_INDEX.get(clip)
    if path is None:
        print(f"ERROR: {clip} not found under {TEST_VIDEO_DIR}")
        sys.exit(1)
    selected.append({"name": name, "clip_id": clip, "path": path})
    print(f"  {name:<32} {clip}")

# ================================================================
# STEP 2: BASELINE MODEL — EfficientNet-B7 + LSTM (fixed weights)
# ================================================================
class EfficientNetB7_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        cnn = efficientnet_b7(weights=None)   # weights don't affect latency
        self.features = cnn.features           # EfficientNet-B7 backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.lstm = nn.LSTM(2560, LSTM_HIDDEN, batch_first=True)
        self.classifier = nn.Linear(LSTM_HIDDEN, NUM_CLASSES)

    def forward(self, frames):                 # frames: (T,3,H,W)
        f = self.pool(self.features(frames)).flatten(1)   # (T,2560)
        out, _ = self.lstm(f.unsqueeze(0))                # (1,T,H)
        return self.classifier(out[:, -1])                # (1,C)

set_seed()                                     # identical weights every run
model = EfficientNetB7_LSTM().to(DEVICE).eval()
baseline_params = sum(p.numel() for p in model.parameters())
w_hash = hashlib.md5(b"".join(p.detach().cpu().numpy().tobytes()
                              for p in model.parameters())).hexdigest()[:12]
print(f"\nBaseline parameters: {baseline_params:,} "
      f"({baseline_params/1e6:.2f}M) | weight hash {w_hash}")

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(frame_rgb):
    img = cv2.resize(frame_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)

@torch.no_grad()
def full_sequence_step(frame_rgb):
    """All T frames through EfficientNet-B7, then LSTM + classifier."""
    t0 = time.perf_counter()
    frames = preprocess(frame_rgb).repeat(T, 1, 1, 1)
    logits = model(frames)
    sync()
    return (time.perf_counter() - t0) * 1000, logits

# ── Load the 5 frames once (identical inputs every run) ─────────
frames_rgb = []
for s in selected:
    cap = cv2.VideoCapture(s["path"]); ret, frame = cap.read(); cap.release()
    if not ret:
        print(f"ERROR reading {s['clip_id']}"); sys.exit(1)
    frames_rgb.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
in_hash = hashlib.md5(b"".join(f.tobytes() for f in frames_rgb)).hexdigest()[:12]
print(f"Input frame hash: {in_hash}")

# ================================================================
# STEP 3: BENCHMARK (repeated N_REPEATS times)
# ================================================================
def summarize(a):
    return dict(mean=np.mean(a), median=np.median(a), std=np.std(a),
                min=np.min(a), max=np.max(a),
                p95=np.percentile(a, 95), p99=np.percentile(a, 99))

def run_once(rep):
    results, out_fps = [], []
    for s, frame_rgb in zip(selected, frames_rgb):
        gc.collect()
        for _ in range(N_WARMUP):
            full_sequence_step(frame_rgb)
        times, ref = [], None
        gc.disable()
        for _ in range(N_TIMED):
            ms, logits = full_sequence_step(frame_rgb)
            times.append(ms)
            out = logits.cpu().numpy()
            if ref is None:
                ref = out
            elif not np.allclose(out, ref, atol=1e-5):
                print(f"  WARNING: output changed within {s['clip_id']}")
        gc.enable()
        times = np.array(times)
        out_fps.append(np.round(ref, 4).tobytes())
        results.append({"snippet": s, "times": times, **summarize(times)})
    out_hash = hashlib.md5(b"".join(out_fps)).hexdigest()[:12]
    return results, out_hash

all_runs = []
for rep in range(1, N_REPEATS + 1):
    print(f"\n── Run {rep}/{N_REPEATS} ──")
    results, out_hash = run_once(rep)
    all_t = np.concatenate([r["times"] for r in results])
    agg = summarize(all_t)
    all_runs.append({"rep": rep, "results": results, "agg": agg,
                     "out_hash": out_hash, "all_t": all_t})
    print(f"  Median={agg['median']:.2f}  P95={agg['p95']:.2f}  "
          f"P99={agg['p99']:.2f} ms | output hash {out_hash}")

# ── Identical-output check across runs ──────────────────────────
hashes = {r["out_hash"] for r in all_runs}
print(f"\nOutputs identical across all runs: "
      f"{'YES' if len(hashes) == 1 else 'NO'} ({', '.join(hashes)})")

# ================================================================
# STEP 4: PER-SNIPPET TABLE (run 1 = reported run), KRUSKAL-WALLIS
# ================================================================
main = all_runs[0]
print(f"\n{'='*86}\n  BASELINE EfficientNet-B7 + LSTM — FULL SEQUENCE (T={T}) — Run 1\n{'='*86}")
print(f"  {'Snippet':<32} {'Mean':>7} {'Median':>7} {'Std':>7} "
      f"{'Min':>7} {'Max':>7} {'P95':>7} {'P99':>7} {'Budget':>6}")
for r in main["results"]:
    print(f"  {r['snippet']['name']:<32} {r['mean']:>7.2f} {r['median']:>7.2f} "
          f"{r['std']:>7.2f} {r['min']:>7.2f} {r['max']:>7.2f} "
          f"{r['p95']:>7.2f} {r['p99']:>7.2f} "
          f"{'PASS' if r['p95'] < TARGET else 'FAIL':>6}")

groups = [r["times"] for r in main["results"]]
H, p = scipy_stats.kruskal(*groups)
meds = [np.median(g) for g in groups]
print(f"  Kruskal-Wallis: H={H:.4f}, p={p:.4f}, "
      f"median spread={max(meds)-min(meds):.2f} ms")
np.save(os.path.join(OUT_DIR, "effnet_full_latencies.npy"), main["all_t"])

# ── Run-to-run variation ────────────────────────────────────────
if N_REPEATS > 1:
    print(f"\n  RUN-TO-RUN VARIATION ({N_REPEATS} runs x 500 trials)")
    for k in ["median", "p95", "p99"]:
        vals = np.array([r["agg"][k] for r in all_runs])
        print(f"  {k.upper():<7} mean={vals.mean():.2f}  sd={vals.std(ddof=1):.2f}  "
              f"range={vals.min():.2f}–{vals.max():.2f} ms  "
              f"CV={vals.std(ddof=1)/vals.mean()*100:.2f}%")

# ================================================================
# STEP 5: COMPARISON WITH PROPOSED MODEL (500 trials each)
# ================================================================
agg = main["agg"]
print(f"\n{'='*86}\n  RQ1 COMPARISON — 500 trials each, same 5 DAiSEE snippets\n{'='*86}")
print(f"  {'Metric':<10} {'Proposed':>10} {'B7+LSTM':>10}")
for k in ["mean", "median", "std", "p95", "p99"]:
    print(f"  {k.upper():<10} {PROPOSED[k]:>10.2f} {agg[k]:>10.2f}")
print(f"  {'Params':<10} {PROPOSED['params']/1e6:>9.2f}M "
      f"{baseline_params/1e6:>9.2f}M")

ratio = agg["median"] / PROPOSED["median"]
verdict = (f"proposed is {ratio:.2f}x faster" if ratio > 1
           else f"baseline is {1/ratio:.2f}x faster")
print(f"  Median ratio: {verdict}")
print(f"  Baseline P95 budget (<{TARGET:.0f} ms): "
      f"{'PASS' if agg['p95'] < TARGET else 'FAIL'}")

pd.DataFrame([
    {"model": "Proposed", **{k: PROPOSED[k] for k in
     ["mean", "median", "std", "p95", "p99"]}, "params": PROPOSED["params"]},
    {"model": "EfficientNet-B7+LSTM (full)", **agg, "params": baseline_params},
]).to_csv(os.path.join(OUT_DIR, "rq1_comparison.csv"), index=False)

# ── Append every run to a permanent log ─────────────────────────
log_rows = [{"timestamp": datetime.now().isoformat(timespec="seconds"),
             "rep": r["rep"], "hardware": HARDWARE_STR, "seed": SEED,
             "threads": N_THREADS, "weight_hash": w_hash, "input_hash": in_hash,
             "output_hash": r["out_hash"],
             **{k: round(r["agg"][k], 2) for k in
                ["mean", "median", "std", "p95", "p99"]}} for r in all_runs]
pd.DataFrame(log_rows).to_csv(RUN_LOG, mode="a",
                              header=not os.path.exists(RUN_LOG), index=False)

print(f"\nSaved: rq1_comparison.csv, effnet_full_latencies.npy, rq1_run_log.csv")
print(f"Hardware: {HARDWARE_STR}")


Hardware: Apple M4, MPS (Metal GPU) | torch 2.10.0 | Python 3.11.14 | seed 42 | threads 4
  Snippet 1: Engaged               5100451035.avi
  Snippet 2: Passive               5100401039.avi
  Snippet 3: Disengaged            5100462003.avi
  Snippet 4: Bright illumination   8264120146.avi
  Snippet 5: Dark illumination     9070011010.mp4

Baseline parameters: 70,084,051 (70.08M) | weight hash e8b4d76e2dca
Input frame hash: d40529ddf091

── Run 1/3 ──
  Median=254.72  P95=272.16  P99=290.03 ms | output hash 19a93c844a85

── Run 2/3 ──
  Median=245.36  P95=258.25  P99=283.86 ms | output hash 19a93c844a85

── Run 3/3 ──
  Median=240.70  P95=244.28  P99=249.12 ms | output hash 19a93c844a85

Outputs identical across all runs: YES (19a93c844a85)

  BASELINE EfficientNet-B7 + LSTM — FULL SEQUENCE (T=10) — Run 1
  Snippet                             Mean  Median     Std     Min     Max     P95     P99 Budget
  Snippet 1: Engaged                253.78  248.91    7.52  244.99  276.90  264.41  26